Instalacion de librerias necesarias para ejecutar el programa

In [ ]:
!pip install --upgrade --force-reinstall --no-deps kaggle
!pip install tqdm
!pip install zipfile

  Using cached kaggle-2.2.1-py3-none-any.whl.metadata (16 kB)
Using cached kaggle-2.2.1-py3-none-any.whl (132 kB)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


0.   **Imports** 

In [ ]:
import zipfile
import os
from collections import Counter
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import matplotlib.pyplot as plt

1.   **Carga** del conjunto de datos

In [ ]:
# Descarga del dataset
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

#Descompresion del dataset
with zipfile.ZipFile("chest-xray-pneumonia.zip", "r") as zip_ref:
    zip_ref.extractall("chest_xray")


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Scripts\kaggle.exe\__main__.py", line 2, in <module>
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\kaggle\__init__.py", line 3, in <module>
    from kaggle.api.kaggle_api_extended import KaggleApi
  File "C:\Users\ASUS\AppData\Local\Programs\Python\Python312\Lib\site-packages\kaggle\api\kaggle_api_extended.py", line 53, in <module>
    from tqdm import tqdm
ModuleNotFoundError: No module named 'tqdm'


2.   **Inspección** del conjunto de datos

In [ ]:
# Ruta base donde están las carpetas train / val / test
base_path = os.path.join(os.environ["USERPROFILE"], "Downloads", "chest_xray", "chest_xray", "chest_xray")

print(base_path)
# Carga del conjunto de entrenamiento
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "train"),
    image_size=(224, 224),
    batch_size=16,
    shuffle=True
)

# Carga del conjunto de validación
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "val"),
    image_size=(224, 224),
    batch_size=16
)

# Carga del conjunto de test
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(base_path, "test"),
    image_size=(224, 224),
    batch_size=16
)

# Muestra las clases detectadas automáticamente
num_classes = len(train_ds.class_names)
class_names = train_ds.class_names
print("Clases del dataset: ", class_names)

# Función para verificar si una imagen está dañada
def es_valida(ruta):
    try:
        Image.open(ruta).verify()
        return True
    except:
        return False

# Recorre train / val / test y elimina imágenes corruptas
for split in os.listdir(base_path):
    if not os.path.isfile(os.path.join(base_path, split)):
        for clase in class_names:
            carpeta = os.path.join(base_path, split, clase)
            for archivo in os.listdir(carpeta):
                ruta = os.path.join(carpeta, archivo)
                if not es_valida(ruta):
                    os.remove(ruta)
                    print("Imagen corrupta o archivo sobrante:", ruta)

# Conteo de imágenes por clase en cada partición
for split in os.listdir(base_path):
    if not os.path.isfile(os.path.join(base_path, split)):
        conteo = {clase: len(os.listdir(os.path.join(base_path, split, clase))) for clase in os.listdir(os.path.join(base_path, split)) if not os.path.isfile(os.path.join(base_path, clase))}
        print(f"Imágenes del conjunto de {split} por clase:", conteo)

C:\Users\ASUS\Downloads\chest_xray\chest_xray\chest_xray
Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
Found 624 files belonging to 2 classes.
Clases del dataset:  ['NORMAL', 'PNEUMONIA']
Imagen corrupta o archivo sobrante: C:\Users\ASUS\Downloads\chest_xray\chest_xray\chest_xray\train\NORMAL\.DS_Store
Imagen corrupta o archivo sobrante: C:\Users\ASUS\Downloads\chest_xray\chest_xray\chest_xray\train\PNEUMONIA\.DS_Store
Imagen corrupta o archivo sobrante: C:\Users\ASUS\Downloads\chest_xray\chest_xray\chest_xray\val\NORMAL\.DS_Store
Imagen corrupta o archivo sobrante: C:\Users\ASUS\Downloads\chest_xray\chest_xray\chest_xray\val\PNEUMONIA\.DS_Store
Imágenes del conjunto de test por clase: {'NORMAL': 234, 'PNEUMONIA': 390}
Imágenes del conjunto de train por clase: {'NORMAL': 1341, 'PNEUMONIA': 3875}
Imágenes del conjunto de val por clase: {'NORMAL': 8, 'PNEUMONIA': 8}


3.   **Acondicionamiento** del conjunto de datos

In [ ]:
# Normalización: escala los píxeles de [0,255] a [0,1]
normalization = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
val_ds = val_ds.map(lambda x, y: (normalization(x), y))
test_ds = test_ds.map(lambda x, y: (normalization(x), y))

# Data augmentation: aumenta variabilidad del dataset
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

# Aplica augmentation solo al conjunto de entrenamiento
train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)

# Optimización
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

ENTRENAMIENTO DE UNA RED NEURONAL CON KERAS DESDE CERO DE EJEMPLO

In [12]:
model = tf.keras.Sequential([
    layers.Input(shape=(224, 224, 3)),

    # Bloque convolucional 1
    layers.Conv2D(32, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    # Bloque convolucional 2
    layers.Conv2D(64, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    # Bloque convolucional 3
    layers.Conv2D(128, (3,3), activation="relu"),
    layers.MaxPooling2D(),

    # Clasificación
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(num_classes, activation="softmax")
])

model.summary()

# ============================
# 4. COMPILACIÓN DEL MODELO
# ============================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",  # Etiquetas enteras
    metrics=["accuracy"]
)

# ============================
# 5. ENTRENAMIENTO
# ============================

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

# Pérdida (loss)
plt.figure(figsize=(10,5))
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Pérdida durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Exactitud (accuracy)
plt.figure(figsize=(10,5))
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Exactitud durante el entrenamiento')
plt.xlabel('Épocas')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


# ============================
# 6. EVALUACIÓN FINAL
# ============================

test_loss, test_acc = model.evaluate(test_ds)
print("Precisión en test:", test_acc)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,218 (42.61 MB)

 Trainable params: 11,169,218 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
326/326 ━━━━━━━━━━━━━━━━━━━━ 93s 282ms/step - accuracy: 0.7414 - loss: 0.5774 - val_accuracy: 0.5000 - val_loss: 0.7568
Epoch 2/5
260/326 ━━━━━━━━━━━━━━━━━━━━ 18s 283ms/step - accuracy: 0.7459 - loss: 0.5723

KeyboardInterrupt: 